# Adding multiple wells

This notebook shows how a WellModel2 can be used to fit multiple wells with one response function. The influence of the individual wells is scaled by the distance to the observation point. 

*Developed by R.C. Caljé, (Artesia Water 2020), D.A. Brakenhoff, (Artesia Water 2019), and R.A. Collenteur, (Artesia Water 2018)*

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pastas as ps
from pastas_plugins.responses.stressmodels import WellModel2
ps.show_versions()

## Load and set data

Set the coordinates of the extraction wells and calculate the distances to the observation well. 

In [ ]:
# Specify coordinates observations
xo = 85850
yo = 383362

# Specify coordinates extractions
relevant_extractions = {
    "Extraction_2": (83588, 383664),
    "Extraction_3": (88439, 382339),
}

# calculate distances
distances = []
for extr, xy in relevant_extractions.items():
    xw = xy[0]
    yw = xy[1]
    distances.append(np.sqrt((xo - xw) ** 2 + (yo - yw) ** 2))

df = pd.DataFrame(
    distances,
    index=relevant_extractions.keys(),
    columns=["Distance to observation well"],
)
df

Read the stresses from their csv files

In [ ]:
# read oseries
oseries = pd.read_csv(
    "data_notebook_10/Observation_well.csv", index_col=0, parse_dates=[0]
).squeeze()
oseries.name = oseries.name.replace(" ", "_")
# read stresses
stresses = {}
for fname in os.listdir("data_notebook_10"):
    series = pd.read_csv(
        os.path.join("data_notebook_10", fname), index_col=0, parse_dates=[0]
    ).squeeze()
    stresses[fname.strip(".csv").replace(" ", "_")] = series

Then plot the observations, together with the different stresses.

In [ ]:
# plot timeseries
f1, axarr = plt.subplots(len(stresses.keys()) + 1, sharex=True, figsize=(10, 8))
oseries.plot(ax=axarr[0], color="k")
axarr[0].set_title(oseries.name)

for i, name in enumerate(stresses.keys(), start=1):
    stresses[name].plot(ax=axarr[i])
    axarr[i].set_title(name)
plt.tight_layout(pad=0)

Get the precipitation and evaporation timeseries and round the index to remove the hours from the timestamps.

In [ ]:
prec = stresses["Precipitation"]
prec.index = prec.index.round("D")
prec.name = "prec"
evap = stresses["Evaporation"]
evap.index = evap.index.round("D")
evap.name = "evap"

Modify the extraction timeseries.

In [ ]:
extraction_ts = {}

for name in relevant_extractions.keys():
    # get extraction timeseries
    s = stresses[name]

    # convert index to end-of-month timeseries
    s.index = s.index.to_period("M").to_timestamp("M")

    # resample to daily values
    new_index = pd.date_range(s.index[0], s.index[-1], freq="D")
    s_daily = ps.ts.timestep_weighted_resample(s, new_index, fast=True).dropna()
    name = name.replace(" ", "_")
    s_daily.name = name

    # append to stresses list
    extraction_ts[name] = s_daily

## Create a model with a separate StressModel for each extraction

First we create a model with a separate StressModel for each groundwater extraction. First we create a model with the heads timeseries and add recharge as a stress.

In [ ]:
# create model
ml = ps.Model(oseries)
rm = ps.RechargeModel(prec, evap, ps.Exponential(), "Recharge")
ml.add_stressmodel(rm)
for name, stress in extraction_ts.items():
    sm = ps.StressModel(stress, ps.Hantush(), name, up=False, settings="well")
    ml.add_stressmodel(sm)

Solve the model.

In [ ]:
ml.solve()

### Visualize the results
Plot the decomposition to see the individual influence of each of the wells.

In [ ]:
ml.plots.results();

## Create a model with a WellModel
We can reduce the number of parameters in the model by including the three extractions in a WellModel. This WellModel takes into account the distances from the three extractions to the observation well, and assumes constant geohydrological properties. All of the extractions now share the same response function, scaled by the distance between the extraction well and the observation well.

In [ ]:
ml_wm = ps.Model(oseries, oseries.name + "_wm")
rm = ps.RechargeModel(prec, evap, ps.Gamma(), "Recharge")
ml_wm.add_stressmodel(rm)
w = ps.WellModel(list(extraction_ts.values()), "WellModel", distances)
ml_wm.add_stressmodel(w)
ml_wm.solve()

In [ ]:
ml_wm.plots.results();

## Create a model with a WellModel2

In [ ]:
ml_wm2 = ps.Model(oseries, oseries.name + "_wm")
rm = ps.RechargeModel(prec, evap, ps.Gamma(), "Recharge")
ml_wm2.add_stressmodel(rm)

w = WellModel2(list(extraction_ts.values()), "WellModel", distances)
ml_wm2.add_stressmodel(w)
ml_wm2.solve()

In [ ]:
ml_wm2.plots.results();